<a href="https://colab.research.google.com/github/Dev-Djelloul/ArtisanConnect---RGPD-Dataset-/blob/master/Outil_IA_analyse_de_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Guidé : Analyse de Sentiments des Avis Clients avec Mistral AI

Bienvenue dans ce notebook ! L'objectif est de vous familiariser avec une application concrète de l'Intelligence Artificielle : l'analyse de sentiments. Nous allons utiliser l'IA de Mistral pour déterminer si les commentaires laissés par les clients d'ArtisanConnect sont plutôt positifs, négatifs ou neutres.

**Votre rôle :**
1.  Lire attentivement les explications.
2.  Exécuter les cellules de code (en cliquant sur le bouton "Play" à gauche de chaque cellule).
3.  Répondre aux quelques questions et compléter les petites sections de code lorsque cela vous sera demandé (cherchez les indications ` complétez_ici `).

Ne vous inquiétez pas si vous n'avez jamais fait de Python, ce notebook est conçu pour vous guider !

## Étape 1 : Qu'est-ce qu'une API et une Clé API ?

Imaginez que vous voulez commander un plat dans un restaurant sans y aller. Vous appelez le restaurant par téléphone :
*   Le **téléphone** est comme l'**API** (Application Programming Interface) : c'est un moyen de communiquer avec un service (le restaurant, ou ici, l'IA de Mistral).
*   Pour que le restaurant sache que c'est bien vous et qu'il peut prendre votre commande, il pourrait vous demander un **mot de passe spécial**. C'est un peu comme la **Clé API**.

Une **Clé API** est une sorte de mot de passe secret que Mistral AI vous donne lorsque vous créez un compte. Elle permet à votre script Python de prouver à Mistral que c'est bien vous qui demandez un service (comme analyser un texte) et que vous avez le droit de le faire.

**IMPORTANT : Une clé API est secrète et personnelle. Ne la partagez jamais publiquement !**

## Étape 2 : Obtenir votre Clé API Mistral

1.  Rendez-vous sur le site de Mistral AI : [https://mistral.ai/](https://mistral.ai/)
2.  Créez un compte gratuit (si vous n'en avez pas déjà un).
3.  Une fois connecté, cherchez la section "API Keys" ou "Clés API" dans votre tableau de bord.
4. Cliquez sur "choose a plan" et prenez l'option "free experiments".
4.  Générez une nouvelle clé API. Copiez-la précieusement.

**Dans Google Colab, pour des raisons de sécurité, nous allons stocker cette clé de manière sécurisée :**
*   Sur le côté gauche de Colab, cliquez sur l'icône en forme de **clé (🔑 Secrets)**.
*   Cliquez sur "+ ADD A NEW SECRET".
*   Dans le champ "Name", écrivez exactement : `MISTRAL_API_KEY`
*   Dans le champ "Value", collez la clé API que vous venez de copier depuis le site de Mistral.
*   Activez l'interrupteur "Notebook access" pour que ce notebook puisse l'utiliser.

Une fois cela fait, la cellule de code suivante pourra récupérer votre clé en toute sécurité.

In [6]:
# Importation de la bibliothèque pour accéder aux secrets de Colab
from google.colab import userdata

# Récupération de la clé API (elle ne sera pas affichée ici par sécurité)
try:
    mistral_api_key = userdata.get('MISTRAL_API_KEY')
    if mistral_api_key:
        print("Clé API Mistral récupérée avec succès depuis les secrets de Colab ! 👍")
        print("Nous pouvons continuer.")
    else:
        print("ATTENTION : La clé API Mistral n'a pas été trouvée dans les secrets.")
        print("Veuillez vérifier l'Étape 2 ci-dessus pour l'ajouter correctement.")
except userdata.SecretNotFoundError:
    print("ATTENTION : Le secret 'MISTRAL_API_KEY' n'existe pas.")
    print("Veuillez vérifier l'Étape 2 ci-dessus pour l'ajouter correctement.")
except Exception as e:
    print(f"Une erreur est survenue : {e}")
    print("Veuillez vérifier l'Étape 2 ci-dessus.")

Clé API Mistral récupérée avec succès depuis les secrets de Colab ! 👍
Nous pouvons continuer.


## Étape 3 : Installer la Bibliothèque Python de Mistral AI

Pour que notre script Python puisse "parler" à l'IA de Mistral, nous avons besoin d'un "traducteur" : une bibliothèque Python spéciale fournie par Mistral.
La cellule suivante va l'installer. L'exécution peut prendre quelques secondes.

In [7]:
# Le '!' au début signifie que c'est une commande à exécuter dans le "terminal" de Colab
!pip install mistralai pandas -q

print("Bibliothèques nécessaires installées !")

Bibliothèques nécessaires installées !


## Étape 4 : Qu'est-ce qu'un "Agent IA" ou un "Modèle" chez Mistral ?

Mistral AI propose différents "cerveaux" d'IA, appelés **modèles** (ou parfois on parle d'agents dans un sens plus large). Chaque modèle a des capacités et des coûts différents.

Pour notre tâche d'analyse de sentiments, nous pouvons utiliser un de leurs modèles performants et polyvalents. Un modèle populaire est `mistral-small-latest`. Il est aussi le moins cher si un jour vous devez opter pour la version payante de Mistral.

Quand nous enverrons un texte à analyser, nous préciserons quel "cerveau" (modèle) nous voulons utiliser.

## Étape 5 : Charger nos Données d'Avis Clients

Nous allons maintenant utiliser la bibliothèque `pandas` (que nous avons installée) pour lire le fichier CSV contenant les avis clients.

**Action requise pour vous :**
1.  Si vous ne l'avez pas encore fait, **uploadez le fichier `avis_clients.csv`** (que nous avons généré précédemment pour le projet ArtisanConnect) dans votre session Google Colab. Pour cela, cliquez sur l'icône dossier (📁) sur le côté gauche, puis sur l'icône "Importer dans le stockage de session" (flèche vers le haut). Vous pouvez également faire un drag & drop si vous préférez.
2.  Assurez-vous que le nom du fichier est bien `avis_clients.csv`.

In [11]:
import pandas as pd

# Tentative de chargement du fichier CSV
try:
    nom_fichier_avis = '/content/avis_clients.csv' # Vous pouvez changer le fichier en mettant son chemin d'accès ici.

    df_avis = pd.read_csv(nom_fichier_avis)
    print(f"Le fichier '{nom_fichier_avis}' a été chargé avec succès !")
    print(f"Il contient {len(df_avis)} avis.")
    print("\nVoici un aperçu des premières lignes de vos données :")
    display(df_avis.head()) # 'display' est mieux que 'print' pour les DataFrames dans Colab

    # --- QUESTION POUR L'ÉTUDIANT ---
    # Regardez l'aperçu ci-dessus. Quelle est, selon vous, la colonne qui contient le texte des commentaires que nous allons analyser ?
    # Écrivez le nom de cette colonne dans la variable ci-dessous.
    # Exemple : si la colonne s'appelle 'TexteDuCommentaire', écrivez : nom_colonne_commentaire = 'TexteDuCommentaire'

    nom_colonne_commentaire = 'commentaire_avis' # Remplacer 'commentaire_avis' par le vrai nom de la colonne

    if nom_colonne_commentaire in df_avis.columns:
        print(f"\nParfait ! Nous allons utiliser la colonne '{nom_colonne_commentaire}' pour l'analyse.")
    else:
        print(f"\nATTENTION : La colonne '{nom_colonne_commentaire}' n'a pas été trouvée. Vérifiez l'aperçu et corrigez son nom.")

except FileNotFoundError:
    print(f"ERREUR : Le fichier '{nom_fichier_avis}' n'a pas été trouvé.")
    print("Veuillez vérifier que vous l'avez bien uploadé dans Colab et que le nom est correct.")
except Exception as e:
    print(f"Une erreur est survenue lors du chargement du fichier : {e}")

Le fichier '/content/avis_clients.csv' a été chargé avec succès !
Il contient 212 avis.

Voici un aperçu des premières lignes de vos données :


,avis_id,produit_id,client_id,commande_id,note_produit,commentaire_avis,date_publication_avis,reponse_artisan_avis,est_avis_verifie,utilite_avis_compteur_positif,utilite_avis_compteur_negatif
0,AVIS-000001,PROD-00321,74b76ccd-b9d1-473d-9c04-011aa946e753,d9a1cda5-65b4-45d2-a4d7-785ed7204989,4,"Absolument ravi de cet achat, je recommande vi...",2024-03-21 10:13:31.777976,NaN,True,29,1
1,AVIS-000002,PROD-00089,10d63abd-0174-4fb2-8347-a059094e236f,8b1a54b1-9e93-4028-b0ce-b970816cdc19,4,"Très déçu par la qualité, ne correspond pas à ...",2024-10-07 17:43:12.234280,NaN,True,0,4
2,AVIS-000003,PROD-00262,92774b37-224d-42ff-bec5-dca1e1e708c5,a252a8bb-4eef-47ea-af5d-3fcb745cd68b,4,"Conforme à mes attentes, sans plus.",2024-10-16 13:11:16.969238,NaN,True,14,2
3,AVIS-000004,PROD-00033,fd114634-ae28-4ae7-b922-403f661b7314,3f410f85-7e7e-41c2-97e4-edc5b40a4084,3,"C'est un produit exceptionnel, bien au-delà de...",2025-03-19 00:34:45.033612,NaN,True,25,2
4,AVIS-000005,PROD-00025,58f31381-c3c3-4202-b30f-739088e6d448,b0e9aabb-9118-46dc-aa51-b391624f6d8b,5,"Qualité médiocre, je ne rachèterai pas.",2025-05-24 17:32:48.472872,Lieu genre personne environ note tel. Froid be...,True,14,1



Parfait ! Nous allons utiliser la colonne 'commentaire_avis' pour l'analyse.


## Étape 6 : Préparer la Connexion à Mistral et Définir notre Tâche

Maintenant que nous avons notre clé API et les bibliothèques, nous pouvons initialiser le "client" Mistral, c'est-à-dire l'objet Python qui va nous permettre de communiquer avec leur IA.

In [12]:
from mistralai import Mistral

try:
  client = Mistral(api_key=mistral_api_key)
  print("Client Mistral initialisé avec succès !")

  # Voici quelques modèles populaires. Le 'small' est le meilleur compromis qualité/coût.
  # D'autres options (peuvent avoir une qualité de réponse différente)
  # 'mistral-medium-latest'
  # 'mistral-large-latest'
  nom_modele_mistral = 'mistral-small-latest' # Vous pouvez changer le modèle ici.

  print(f"Nous allons utiliser le modèle : {nom_modele_mistral}")

except Exception as e:
    print(f"Erreur lors de l'initialisation du client Mistral : {e}")
    print("Assurez-vous que votre clé API est correcte et active.")


Client Mistral initialisé avec succès !
Nous allons utiliser le modèle : mistral-small-latest


### Définir le "Prompt" pour l'Analyse de Sentiments

Un **prompt**, c'est l'instruction que l'on donne à l'IA. Pour l'analyse de sentiments, nous allons lui demander de lire un commentaire et de nous dire s'il est "Positif", "Négatif", ou "Neutre".

Il est important de bien formuler ce prompt. Voici un exemple :

In [15]:
commentaire = "Incroyable ! La meilleure acquisition de l'année."

prompt = f"""
Analyse l'avis client suivant:
{commentaire}
Ta réponse ne doit comporter qu'un mot parmi "Positif", "Négatif" ou "Neutre". C'est tout.
"""

## Étape 7 : Analyser le Sentiment d'UN Seul Avis

Avant d'analyser tous les avis, testons avec un seul pour voir comment ça marche.

In [16]:
try:
    print(f"Commentaire à analyser : \n---\n{commentaire}\n---")

    # Envoi de la requête à Mistral
    # Note: 'System' message can set the context for the AI, 'User' message is our specific request.
    chat_response = client.chat.complete(
        model=nom_modele_mistral,
        messages=[
            {"role" : "user",
             "content" : prompt
            }
        ],
        temperature=0.1 # Basse température pour des réponses plus déterministes/factuelles
    )

    sentiment_predit = chat_response.choices[0].message.content.strip()
    print(f"\nSentiment prédit par Mistral : {sentiment_predit}")

except Exception as e:
    print(f"Une erreur est survenue lors de l'analyse du commentaire : {e}")
    print("Vérifiez votre connexion internet et votre clé API.")


Commentaire à analyser : 
---
Incroyable ! La meilleure acquisition de l'année.
---

Sentiment prédit par Mistral : Positif


### Votre Réflexion sur le Premier Test :

Lors du premier test, la réponse produite par Mistral correspond globalement au sens du commentaire analysé : le sentiment retourné (“Positif”, “Neutre” ou “Négatif”) reflète correctement l’impression générale exprimée dans le texte. Ce test unitaire est utile car il permet de vérifier rapidement que l’appel à l’API fonctionne, que le prompt est compris, et que le format de sortie est conforme à ce qui est attendu avant de lancer l’analyse sur plusieurs avis.

J’ai remarqué que le paramètre temperature=0.1 contribue à rendre la sortie plus stable et plus reproductible. Une température faible réduit la part d’aléatoire dans les réponses du modèle : l’IA est moins “créative” et plus “déterministe”, ce qui est préférable pour une tâche de classification. Dans le cadre d’ArtisanConnect, cet aspect est important car on cherche une prédiction cohérente d’un avis à l’autre, et non des variations de sentiment dues au hasard.

Malgré cette stabilité, ce premier test rappelle aussi une limite importante : l’IA peut se tromper sur des formulations ambiguës, notamment l’ironie et le sarcasme. Par exemple, un commentaire qui contient des mots positifs mais dont le sens réel est négatif (“Super… encore un colis abîmé”) peut être difficile à interpréter automatiquement. Cela justifie le fait de garder un contrôle humain pour les cas douteux, ou de prévoir un indicateur “à vérifier” lorsque la confiance est faible.

Enfin, le fait de forcer une sortie très contrainte (un seul mot parmi “Positif / Neutre / Négatif”) est un bon choix pour ce projet. Cette contrainte réduit les réponses hors-sujet, facilite l’automatisation (stockage en base, filtrage, statistiques) et rend l’intégration dans un tableau de bord plus simple. En contrepartie, elle simplifie la réalité : certains avis peuvent contenir plusieurs émotions ou concerner plusieurs sujets (produit vs livraison). Une amélioration possible serait d’ajouter, en plus du sentiment, un champ “motif principal” (livraison, qualité, service, prix), mais pour un Proof Of Concept (POC) la classification en 3 classes est largement adaptée.

## Étape 8 : Analyser le Sentiment pour Plusieurs Avis

Maintenant que le test sur un avis a fonctionné, nous allons appliquer cela à un petit échantillon de nos avis (par exemple, les 5 premiers) pour voir comment cela se comporte sur plusieurs textes. Analyser tout le dataset pourrait prendre du temps et utiliser beaucoup de "crédits" sur votre compte Mistral gratuit.

Nous allons créer une fonction pour rendre le code plus propre.

In [18]:
from time import sleep

def analyser_sentiment_mistral(commentaire, client_mistral, modele):
    """Analyse le sentiment d'un texte donné en utilisant Mistral AI."""

    prompt = f"""
Analyse l'avis client suivant:
{commentaire}
Ta réponse ne doit comporter qu'un mot parmi "Positif", "Négatif" ou "Neutre". C'est tout.
"""
    try:
        chat_response = client.chat.complete(
        model=nom_modele_mistral,
        messages=[
                {"role" : "user",
                "content" : prompt
                }
            ],
            temperature=0.1 # Basse température pour des réponses plus déterministes/factuelles
        )
        sentiment = chat_response.choices[0].message.content.strip()
        # Sécurité : s'assurer que la réponse est bien l'une des catégories attendues

        # Avec un plan gratuit, on ne peut faire que 1 demande à l'IA par sec.
        # Donc on attends (sleep) pendant 1,5 sec avant de faire plus.
        sleep(1.5)

        if sentiment not in ['Positif', 'Négatif', 'Neutre']:
            return "Réponse IA inattendue"
        return sentiment
    except Exception as e:
        print(f"Erreur pendant l'analyse pour le commentaire : {commentaire[:50]}... Erreur : {e}")
        return "Erreur d'analyse"


# On peut prendre 5 avis pour faire un test simple
nombre_avis_a_analyser = 5 # Vous pouvez bien sûr changer le nombre d'avis ici.

df_echantillon = df_avis.head(nombre_avis_a_analyser).copy()

# Ajout d'une nouvelle colonne pour stocker les sentiments
# On utilise .loc pour être sûr de modifier le DataFrame df_echantillon
df_echantillon.loc[:, 'sentiment_predit_mistral'] = df_echantillon[nom_colonne_commentaire].apply(
    lambda commentaire: analyser_sentiment_mistral(commentaire, client, nom_modele_mistral)
)

print("\nVoici l'échantillon d'avis avec les sentiments prédits :")
display(df_echantillon[[nom_colonne_commentaire, 'sentiment_predit_mistral']])


Voici l'échantillon d'avis avec les sentiments prédits :


,commentaire_avis,sentiment_predit_mistral
0,"Absolument ravi de cet achat, je recommande vi...",Positif
1,"Très déçu par la qualité, ne correspond pas à ...",Négatif
2,"Conforme à mes attentes, sans plus.",Neutre
3,"C'est un produit exceptionnel, bien au-delà de...",Positif
4,"Qualité médiocre, je ne rachèterai pas.",Négatif


### Vos Réflexions sur l'Analyse de l'Échantillon :

1) La cohérence des prédictions

Sur l’échantillon testé, les prédictions de sentiment produites par Mistral sont globalement cohérentes avec le contenu des commentaires. Cependant, j’observe aussi des cas ambigus et au moins une discordance notable entre la note attribuée (1–5) et le sentiment déduit du texte. Cette situation m’amène à conclure que, dans ce dataset, la note n’est pas toujours un indicateur fiable de la satisfaction réelle exprimée dans le commentaire.

En analysant plusieurs lignes, je constate que le fichier avis_clients contient des incohérences “note ↔ commentaire” (par exemple une note élevée associée à un texte critique, ou l’inverse). Cela signifie que la note ne peut pas être utilisée comme “vérité terrain” (label) pour mesurer la performance de l’IA. Autrement dit, je ne peux pas conclure “l’IA se trompe” uniquement parce que sa prédiction ne correspond pas à la note, puisque la note elle-même peut être erronée ou imprécise.

J’en déduis que l’IA semble principalement classer le contenu textuel (ce qui est logique, puisque le prompt lui fournit surtout le commentaire). Cette approche est pertinente pour détecter l’émotion et le ressenti, mais elle peut contredire la note si celle-ci est mal saisie, si le commentaire est ironique, ou si l’avis mélange plusieurs sujets (ex. produit satisfaisant mais livraison problématique). Pour fiabiliser l’usage en production, il serait pertinent de mettre en place une validation humaine sur un petit sous-ensemble d’avis (échantillon annoté manuellement) afin de disposer d’une référence plus robuste, ou d’ajouter une règle de résolution des contradictions.

Enfin, une mitigation simple consiste à détecter automatiquement les cas incohérents (ex. note 5 mais sentiment “Négatif”) et à les marquer “à vérifier”. Cela permet d’éviter de déclencher des actions automatiques sur des cas ambiguës et de réserver la relecture humaine aux avis “à risque”.

2) L'utilisation faite par ArtisanConnect

Pour ArtisanConnect, l’analyse de sentiment apporte surtout un gain opérationnel : elle transforme des avis textuels non structurés en informations rapidement exploitables par les équipes. L’objectif n’est pas de remplacer l’humain, mais d’aider le support et les équipes qualité à prioriser les retours et à réagir plus vite aux problèmes qui peuvent impacter la réputation de la marketplace.

Un premier usage concret est le tri automatique des avis. Grâce à la colonne sentiment_predit_mistral, ArtisanConnect peut regrouper les avis “Négatifs” dans une file dédiée, afin qu’ils soient traités en priorité. Cette priorisation permet de réduire le temps de réponse et d’améliorer l’expérience client, par exemple en visant un délai de réponse (24–48h) pour les avis négatifs et en suivant ensuite ce KPI dans le pilotage.

Un deuxième usage est l’alerte interne. Si plusieurs avis négatifs concernent le même produit, la même catégorie, ou le même artisan sur une courte période, ArtisanConnect peut déclencher une alerte vers l’équipe support/qualité. Cela permet d’identifier rapidement des incidents récurrents (ex. casse à la livraison, défaut de fabrication, description produit trompeuse, retard de production, problème transporteur) et de corriger la cause avant que le problème ne s’amplifie.

Un troisième usage est l’amélioration continue. En agrégeant les sentiments par catégorie ou par produit, ArtisanConnect peut repérer des tendances et décider d’actions correctives : améliorer les consignes d’emballage, renforcer les standards qualité, clarifier certaines fiches produit, ou accompagner les artisans sur les points qui génèrent le plus de retours négatifs. La valeur de l’outil IA se situe donc autant dans le suivi des indicateurs que dans l’aide à la décision opérationnelle.

Enfin, les résultats doivent être utilisés avec prudence. Les erreurs possibles (ironie, ambiguïté, contradictions note/commentaire) montrent qu’il est préférable de considérer l’IA comme une aide au tri. Une bonne pratique consiste à prévoir une revue manuelle pour les cas incohérents ou incertains (par exemple, avis prédit “Négatif” avec une note 5). Cette combinaison “automatisation + contrôle” permet d’obtenir des gains de productivité tout en limitant les risques d’erreurs.

3) Les points de vigilance RGPD (à respecter explicitement)

Concernant la protection des données, il est essentiel de respecter le principe de minimisation : pour l’analyse IA, ArtisanConnect doit limiter les données envoyées à l’API au texte de l’avis (et éventuellement un identifiant technique non directement identifiant), et ne jamais transmettre de données sensibles ou identifiantes telles que l’email, le téléphone, l’adresse, l’IBAN, etc. Cette précaution réduit le risque de fuite de données et améliore la conformité RGPD dès la phase de Proof Of Concept (POC).

In [19]:
# Export de l'échantillon enrichi (utile pour preuve / partage / Dashboard Looker Studio)
df_echantillon.to_csv("avis_echantillon_avec_sentiment.csv", index=False)
print("Export OK : avis_echantillon_avec_sentiment.csv")


Export OK : avis_echantillon_avec_sentiment.csv


In [20]:

# Affiche les 10 premières lignes du DataFrame df_echantillon contenant la nouvelle colonne "sentiment_predit_mistral"
df_echantillon.head(10)


,avis_id,produit_id,client_id,commande_id,note_produit,commentaire_avis,date_publication_avis,reponse_artisan_avis,est_avis_verifie,utilite_avis_compteur_positif,utilite_avis_compteur_negatif,sentiment_predit_mistral
0,AVIS-000001,PROD-00321,74b76ccd-b9d1-473d-9c04-011aa946e753,d9a1cda5-65b4-45d2-a4d7-785ed7204989,4,"Absolument ravi de cet achat, je recommande vi...",2024-03-21 10:13:31.777976,NaN,True,29,1,Positif
1,AVIS-000002,PROD-00089,10d63abd-0174-4fb2-8347-a059094e236f,8b1a54b1-9e93-4028-b0ce-b970816cdc19,4,"Très déçu par la qualité, ne correspond pas à ...",2024-10-07 17:43:12.234280,NaN,True,0,4,Négatif
2,AVIS-000003,PROD-00262,92774b37-224d-42ff-bec5-dca1e1e708c5,a252a8bb-4eef-47ea-af5d-3fcb745cd68b,4,"Conforme à mes attentes, sans plus.",2024-10-16 13:11:16.969238,NaN,True,14,2,Neutre
3,AVIS-000004,PROD-00033,fd114634-ae28-4ae7-b922-403f661b7314,3f410f85-7e7e-41c2-97e4-edc5b40a4084,3,"C'est un produit exceptionnel, bien au-delà de...",2025-03-19 00:34:45.033612,NaN,True,25,2,Positif
4,AVIS-000005,PROD-00025,58f31381-c3c3-4202-b30f-739088e6d448,b0e9aabb-9118-46dc-aa51-b391624f6d8b,5,"Qualité médiocre, je ne rachèterai pas.",2025-05-24 17:32:48.472872,Lieu genre personne environ note tel. Froid be...,True,14,1,Négatif


In [27]:
def sentiment_depuis_note(note):
    if note <= 2:
        return "Négatif"
    elif note == 3:
        return "Neutre"
    else:
        return "Positif"

df_echantillon["sentiment_note"] = df_echantillon["note_produit"].apply(sentiment_depuis_note)

df_echantillon["flag_incoherence_note_vs_ia"] = (
    (df_echantillon["sentiment_note"].isin(["Positif","Négatif"])) &
    (df_echantillon["sentiment_predit_mistral"].isin(["Positif","Négatif"])) &
    (df_echantillon["sentiment_note"] != df_echantillon["sentiment_predit_mistral"])
)

df_echantillon[[
    "avis_id","note_produit","commentaire_avis",
    "sentiment_note","sentiment_predit_mistral","flag_incoherence_note_vs_ia"
]].head(20)


,avis_id,note_produit,commentaire_avis,sentiment_note,sentiment_predit_mistral,flag_incoherence_note_vs_ia
0,AVIS-000001,4,"Absolument ravi de cet achat, je recommande vi...",Positif,Positif,False
1,AVIS-000002,4,"Très déçu par la qualité, ne correspond pas à ...",Positif,Négatif,True
2,AVIS-000003,4,"Conforme à mes attentes, sans plus.",Positif,Neutre,False
3,AVIS-000004,3,"C'est un produit exceptionnel, bien au-delà de...",Neutre,Positif,False
4,AVIS-000005,5,"Qualité médiocre, je ne rachèterai pas.",Positif,Négatif,True


In [28]:
df_echantillon["flag_incoherence_note_vs_ia"].value_counts()


,count
flag_incoherence_note_vs_ia,
False,3
True,2


### Vos Réflexions sur l'Incohérence Note vs IA (Note de produit / sentiment Note et Prédiction Mistral) :

Dans ce dataset, j’observe que la note (1 à 5) et le contenu du commentaire ne sont pas toujours cohérents. Cela signifie que la note ne peut pas être considérée comme une “vérité terrain” fiable pour évaluer automatiquement si l’IA se trompe. Une discordance “note vs IA” peut venir d’une erreur de saisie, d’un avis qui mélange plusieurs sujets (produit satisfaisant mais livraison problématique), ou d’un commentaire ambigu (ironie / sarcasme).

Pour analyser ce point, j’ai créé une variable sentiment_note en traduisant la note en catégories simples (1–2 = Négatif, 3 = Neutre, 4–5 = Positif), puis j’ai comparé ce sentiment issu de la note à la prédiction sentiment_predit_mistral. J’ai ensuite ajouté un indicateur flag_incoherence_note_vs_ia afin d’identifier les cas où la note et l’IA divergent fortement. J’ai volontairement limité ce flag aux cas Positif vs Négatif, afin d’éviter de sur-interpréter les avis “Neutres”, souvent plus difficiles à trancher.

D’un point de vue métier, ces incohérences sont utiles : elles permettent à ArtisanConnect de repérer des avis “à vérifier” et de déclencher une revue humaine prioritaire. En production, ce mécanisme peut servir de garde-fou : plutôt que de prendre une décision automatique sur un cas incertain, la marketplace peut mettre l’avis en “contrôle qualité” (modération/support) avant de conclure. Cela renforce la fiabilité du tri automatique et limite le risque d’actions incorrectes (ex. alerte inutile, mauvaise priorisation, ou réponse inadaptée).

Enfin, cette étape met en évidence un point clé : pour évaluer réellement la performance de l’IA, il faut un jeu d’avis annotés manuellement (même petit), car la note brute ne reflète pas toujours le sentiment réel exprimé dans le texte.

## Étape 9 : Conclusion et Prochaines Étapes

Félicitations ! Vous venez d'utiliser une IA de pointe (Mistral AI) pour effectuer une analyse de sentiments sur des données réelles (ou presque !).

**Ce que nous avons appris ensemble :**
*   Ce qu'est une API et une clé API.
*   Comment interagir avec un modèle d'IA comme ceux de Mistral.
*   Comment formuler un "prompt" pour obtenir le résultat souhaité.
*   Un exemple concret d'application de l'IA : l'analyse de sentiments.

**Pour le projet ArtisanConnect, cette analyse de sentiments pourrait être très utile pour :**
*   Identifier rapidement les clients mécontents et essayer de résoudre leurs problèmes.
*   Comprendre quels types de produits ou quels artisans reçoivent les meilleurs (ou les pires) retours.
*   Mesurer la satisfaction client globale au fil du temps.

Les résultats de cette analyse (la colonne `sentiment_predit_mistral`) pourraient ensuite être utilisés dans votre **Tableau de Bord Looker Studio (Étape 5)** pour créer des visualisations !

**Points importants à retenir :**
*   L'IA est un outil puissant, mais elle n'est pas magique. La qualité du "prompt" est essentielle.
*   Il faut toujours garder un regard critique sur les résultats de l'IA.
*   L'utilisation d'API a souvent un coût (même si Mistral propose un accès gratuit pour commencer). Il faut en tenir compte pour des applications à grande échelle.
*   La gestion des données personnelles (RGPD) reste importante même quand on utilise une IA.

Merci d'avoir suivi ce notebook guidé !